# S02 Solutions — Control Flow, Functions & Debugger

**Python for Applied Physics** · Session 2 solutions  

> These solutions represent one correct approach. Your code may differ and still be perfectly valid.

---

## Solution 1 — Laser safety classification

In [ ]:
def laser_class(P_avg_W, wavelength_nm):
    """
    Return simplified IEC laser safety class.

    Parameters
    ----------
    P_avg_W : float
        Average power [W].
    wavelength_nm : float
        Wavelength [nm].

    Returns
    -------
    str
        Safety class string.
    """
    is_visible = 400 <= wavelength_nm <= 700

    if P_avg_W < 0.39e-3:
        return "Class 1"
    elif is_visible and P_avg_W < 1e-3:
        return "Class 2"
    elif P_avg_W < 5e-3:
        return "Class 3R"
    elif P_avg_W < 0.5:
        return "Class 3B"
    else:
        return "Class 4"


test_cases = [
    (0.2e-3,  532,  "green pointer"),
    (3e-3,    650,  "alignment laser"),
    (100e-3,  980,  "CW diode"),
    (5.0,    1030,  "Yb:KGW oscillator"),
]

print(f"{'Laser':<22} {'P (mW)':>8}  {'λ (nm)':>7}  Class")
print("-" * 52)
for P, lam, name in test_cases:
    cls = laser_class(P, lam)
    print(f"{name:<22} {P*1e3:>8.1f}  {lam:>7d}  {cls}")

## Solution 2 — Wavelength–frequency–wavenumber table

In [ ]:
def spectral_table(wavelengths_nm):
    """
    Print a spectral conversion table for a list of wavelengths.

    Parameters
    ----------
    wavelengths_nm : list of float
        Wavelengths in nm.
    """
    c = 3e8           # m/s
    h = 6.626e-34     # J·s
    e = 1.602e-19     # C

    print(f"{'λ (nm)':>8}  {'ν (THz)':>10}  {'ν̃ (cm⁻¹)':>11}  {'E (eV)':>8}")
    print("-" * 46)
    for lam_nm in wavelengths_nm:
        lam_m  = lam_nm * 1e-9
        lam_cm = lam_nm * 1e-7
        nu_THz = c / lam_m / 1e12
        nu_cm  = 1.0 / lam_cm
        E_eV   = h * (c / lam_m) / e
        print(f"{lam_nm:>8.0f}  {nu_THz:>10.2f}  {nu_cm:>11.1f}  {E_eV:>8.4f}")


spectral_table([355, 515, 532, 800, 1030, 1064, 1550])

## Solution 3 — Fresnel equations

In [ ]:
import math

def fresnel_reflectance(n1, n2, theta_i_deg):
    """
    Fresnel reflectance for s- and p-polarizations.

    Parameters
    ----------
    n1, n2 : float
        Refractive indices of incident and transmitted media.
    theta_i_deg : float
        Angle of incidence [degrees].

    Returns
    -------
    R_s, R_p : float
        Reflectance for s- and p-polarizations.
    """
    theta_i = math.radians(theta_i_deg)
    sin_t   = n1 * math.sin(theta_i) / n2

    if abs(sin_t) > 1.0:   # total internal reflection
        return 1.0, 1.0

    theta_t  = math.asin(sin_t)
    cos_i    = math.cos(theta_i)
    cos_t    = math.cos(theta_t)

    r_s = (n1 * cos_i - n2 * cos_t) / (n1 * cos_i + n2 * cos_t)
    r_p = (n2 * cos_i - n1 * cos_t) / (n2 * cos_i + n1 * cos_t)

    return r_s**2, r_p**2


# Part 2: table for glass
n1, n2 = 1.0, 1.52
print(f"{'θ (°)':>6}  {'R_s':>8}  {'R_p':>8}  {'R_avg':>8}")
print("-" * 36)
for angle in range(0, 85, 10):
    Rs, Rp = fresnel_reflectance(n1, n2, angle)
    print(f"{angle:>6}  {Rs:>8.4f}  {Rp:>8.4f}  {(Rs+Rp)/2:>8.4f}")

# Part 3: Brewster angle
print()
min_Rp = 1.0
brewster = 0.0
for angle_10th in range(0, 900):
    angle = angle_10th * 0.1
    _, Rp = fresnel_reflectance(n1, n2, angle)
    if Rp < min_Rp:
        min_Rp = Rp
        brewster = angle

print(f"Brewster angle (numerical): {brewster:.1f}°")
print(f"Brewster angle (analytic):  {math.degrees(math.atan(n2/n1)):.1f}°")

## Solution 4 — Pulse train energy budget

In [ ]:
import math

def pulse_train_stats(energies_uJ, threshold_uJ):
    """
    Statistical summary of a pulse energy dataset.

    Parameters
    ----------
    energies_uJ : list of float
        Pulse energies [µJ].
    threshold_uJ : float
        Count pulses exceeding this energy [µJ].

    Returns
    -------
    dict
        Keys: mean, std, min, max, rms_noise_pct, n_above_threshold.
    """
    n = len(energies_uJ)
    mean = sum(energies_uJ) / n

    variance = sum((e - mean)**2 for e in energies_uJ) / (n - 1)   # sample std
    std = math.sqrt(variance)

    E_min = min(energies_uJ)
    E_max = max(energies_uJ)
    rms_noise = std / mean * 100
    n_above = sum(1 for e in energies_uJ if e > threshold_uJ)

    return {
        "mean": mean,
        "std": std,
        "min": E_min,
        "max": E_max,
        "rms_noise_pct": rms_noise,
        "n_above_threshold": n_above,
    }


pulse_energies_uJ = [48.2, 51.1, 49.8, 50.3, 47.9, 52.0, 50.7, 49.5, 51.3, 48.8,
                     50.1, 49.2, 51.8, 50.5, 48.6, 51.0, 49.9, 50.8, 47.5, 52.3]

stats = pulse_train_stats(pulse_energies_uJ, threshold_uJ=51.5)

print("Pulse train statistics")
print("-" * 30)
print(f"  Mean energy      : {stats['mean']:.2f} µJ")
print(f"  Std deviation    : {stats['std']:.3f} µJ")
print(f"  Min / Max        : {stats['min']:.1f} / {stats['max']:.1f} µJ")
print(f"  RMS noise        : {stats['rms_noise_pct']:.2f} %")
print(f"  Shots > 51.5 µJ  : {stats['n_above_threshold']}")

## Solution 5 — Beam propagation with ABCD matrices

In [ ]:
import math

def apply_abcd(q, M):
    """
    Apply a 2×2 ABCD matrix to a complex beam parameter q.

    Parameters
    ----------
    q : complex
        Input beam parameter.
    M : list of list
        2×2 ABCD matrix [[A,B],[C,D]].

    Returns
    -------
    complex
        Output beam parameter.
    """
    A, B = M[0][0], M[0][1]
    C, D = M[1][0], M[1][1]
    return (A * q + B) / (C * q + D)


def propagate_system(q0, elements):
    """
    Propagate beam parameter through a sequence of optical elements.

    Parameters
    ----------
    q0 : complex
        Input complex beam parameter.
    elements : list of (str, float)
        Each tuple is ('free', distance_m) or ('lens', focal_length_m).

    Returns
    -------
    list of complex
        Beam parameter after each element (includes initial q0).
    """
    q_history = [q0]
    q = q0
    for elem_type, param in elements:
        if elem_type == 'free':
            d = param
            M = [[1, d], [0, 1]]
        elif elem_type == 'lens':
            f = param
            M = [[1, 0], [-1/f, 1]]
        else:
            raise ValueError(f"Unknown element type: {elem_type}")
        q = apply_abcd(q, M)
        q_history.append(q)
    return q_history


def beam_waist_from_q(q, wavelength):
    """
    Extract 1/e² beam radius from complex beam parameter.

    Parameters
    ----------
    q : complex
        Complex beam parameter [m].
    wavelength : float
        Laser wavelength [m].

    Returns
    -------
    float
        Beam radius w [m].
    """
    inv_q = 1.0 / q
    # w = sqrt(-lambda / (pi * Im(1/q)))
    return math.sqrt(-wavelength / (math.pi * inv_q.imag))


# System definition
lam = 1030e-9
w0  = 1e-3       # 1 mm beam waist
z_R = math.pi * w0**2 / lam
q0  = complex(0, z_R)   # at waist: q = i·z_R

elements = [
    ('free', 0.500),   # 500 mm free space
    ('lens', 0.200),   # f = 200 mm lens
    ('free', 0.300),   # 300 mm free space
]

q_list = propagate_system(q0, elements)

labels = ["Input waist", "After 500 mm", "After lens f=200mm", "After +300 mm"]
print(f"{'Position':<22}  {'w (µm)':>10}")
print("-" * 36)
for label, q in zip(labels, q_list):
    w = beam_waist_from_q(q, lam)
    print(f"{label:<22}  {w*1e6:>10.1f}")

## Exercise 6 🟢 — Wavelength Grid


In [ ]:
# Create 11 evenly spaced wavelengths from 700 to 900 nm
wavelengths = [700 + i * 20 for i in range(11)]

c = 2.998e8  # m/s

print(f"{'λ (nm)':>10s}  {'ν (THz)':>10s}")
print("-" * 23)
for lam in wavelengths:
    nu_THz = c / (lam * 1e-9) / 1e12
    print(f"{lam:10.1f}  {nu_THz:10.1f}")

# Closest to 800 nm
target = 800
closest = min(wavelengths, key=lambda w: abs(w - target))
print(f"Closest to {target} nm: {closest} nm")

## Solution 7 — Comprehensions: spectral filter catalog

In [ ]:
filters = [
    {"name": "BP532",  "center_nm": 532,  "bw_nm": 10,  "T_peak": 0.92},
    {"name": "BP800",  "center_nm": 800,  "bw_nm": 40,  "T_peak": 0.88},
    {"name": "LP1000", "center_nm": 1000, "bw_nm": None, "T_peak": 0.95},
    {"name": "BP1030", "center_nm": 1030, "bw_nm": 5,   "T_peak": 0.85},
    {"name": "BP1550", "center_nm": 1550, "bw_nm": 12,  "T_peak": 0.91},
    {"name": "SP700",  "center_nm": 700,  "bw_nm": None, "T_peak": 0.80},
]

# 1. Bandpass filter names
bandpass_names = [f["name"] for f in filters if f["bw_nm"] is not None]
print("Bandpass filters:", bandpass_names)

# 2. High-transmission filters, sorted by center wavelength
high_T = sorted(
    [(f["name"], f["center_nm"]) for f in filters if f["T_peak"] > 0.90],
    key=lambda x: x[1]
)
print("High-T filters (sorted by λ):", high_T)

# 3. Name → center frequency (THz)
c = 3e8
freq_dict = {f["name"]: c / (f["center_nm"] * 1e-9) / 1e12 for f in filters}
print("Center frequencies (THz):")
for name, nu in freq_dict.items():
    print(f"  {name}: {nu:.1f} THz")

## Solution 8 — B-integral (nonlinear phase accumulation)

In [ ]:
import math

def b_integral_numerical(I0, n2, g, L, wavelength, N=1000):
    """
    B-integral computed with the trapezoidal rule.

    Parameters
    ----------
    I0 : float        Peak intensity at z=0 [W/m²].
    n2 : float        Nonlinear refractive index [m²/W].
    g  : float        Net gain coefficient [m⁻¹]. 0 for passive medium.
    L  : float        Medium length [m].
    wavelength : float  Laser wavelength [m].
    N  : int          Number of integration steps.

    Returns
    -------
    float
        B-integral [rad].
    """
    dz = L / N
    total = 0.0
    for k in range(N):
        z0 = k * dz
        z1 = (k + 1) * dz
        I0_ = I0 * math.exp(g * z0)
        I1_ = I0 * math.exp(g * z1)
        total += 0.5 * (I0_ + I1_) * dz   # trapezoidal
    return 2 * math.pi / wavelength * n2 * total


def b_integral_analytic(I0, n2, g, L, wavelength):
    """
    Analytic B-integral.

    B = (2π/λ) · n2 · I0 · (exp(g·L) - 1) / g   for g ≠ 0
    B = (2π/λ) · n2 · I0 · L                      for g = 0
    """
    prefactor = 2 * math.pi / wavelength * n2 * I0
    if abs(g) < 1e-12:
        integral = L
    else:
        integral = (math.exp(g * L) - 1) / g
    return prefactor * integral


# Test cases
params = [
    {"name": "Fused silica (passive)", "n2": 2.5e-20, "g": 0,   "L": 1e-3},
    {"name": "Yb:YAG gain crystal",   "n2": 2.5e-20, "g": 5.0, "L": 1e-3},
]
I0  = 1e13      # W/m²
lam = 1030e-9   # m

for p in params:
    B_num = b_integral_numerical(I0, p["n2"], p["g"], p["L"], lam)
    B_ana = b_integral_analytic( I0, p["n2"], p["g"], p["L"], lam)
    print(f"{p['name']}")
    print(f"  Numerical: {B_num:.5f} rad  |  Analytic: {B_ana:.5f} rad")
    print()

# Part 4: find I0 where B > pi (passive fused silica)
n2  = 2.5e-20
L   = 1e-3
I_test = 1e12
step    = 1e12
while b_integral_analytic(I_test, n2, 0, L, lam) < math.pi:
    I_test += step

print(f"B exceeds π at I0 ≈ {I_test:.2e} W/m² ({I_test/1e4:.2e} W/cm²)")

## Solution 9 — Debugging challenge (three bugs fixed)

In [ ]:
import math

def transform_limit(delta_lambda_nm, center_lambda_nm):
    """
    Transform-limited pulse duration for a Gaussian pulse.

    Parameters
    ----------
    delta_lambda_nm : float
        Spectral bandwidth FWHM [nm].
    center_lambda_nm : float
        Center wavelength [nm].

    Returns
    -------
    float
        Transform-limited pulse duration FWHM [fs].
    """
    TBP = 0.4413
    c   = 3e8

    dl = delta_lambda_nm * 1e-9
    # FIX 1: center_lambda_nm was not converted to metres
    lc = center_lambda_nm * 1e-9

    # FIX 2: Δν = (c / λ²) · Δλ  — was missing the **2 on λ
    d_nu = (c / lc**2) * dl

    # FIX 3: Δt = TBP / Δν  — was TBP * Δν (dimensions wrong)
    dt_s = TBP / d_nu

    return dt_s * 1e15


print(f"10 nm @ 800 nm  → {transform_limit(10, 800):.1f} fs  (expected ~94 fs)")
print(f" 5 nm @ 1030 nm → {transform_limit(5, 1030):.1f} fs  (expected ~283 fs)")

# Summary of bugs:
# Bug 1: lc = center_lambda_nm  →  forgot to multiply by 1e-9  (nm → m)
# Bug 2: d_nu = (c / lc) * dl   →  should be (c / lc**2)  (correct derivative dν/dλ)
# Bug 3: dt_s = TBP * d_nu       →  should be TBP / d_nu   (TBP = Δt · Δν → Δt = TBP / Δν)